# LeetCode 281 — Zigzag Iterator bằng Python

Link đề: https://leetcode.com/problems/zigzag-iterator/ · Độ khó: **Medium** · Chủ đề: Design, Queue, Array, Iterator

Skeleton LeetCode:

```python
class ZigzagIterator:
    def __init__(self, v1: List[int], v2: List[int]):

    def next(self) -> int:

    def hasNext(self) -> bool:


# Your ZigzagIterator object will be instantiated and called as such:
# i, v = ZigzagIterator(v1, v2), []
# while i.hasNext(): v.append(i.next())
```

Ghi chú: đây là bài **Premium** của LeetCode; đề bài viết theo hiểu biết về bài (và bản mô tả đã tải), nên đối chiếu lại nếu có tài khoản Premium. Mọi đáp án trong notebook đều được kiểm chứng bằng code: ta tự dựng khung test (gọi `hasNext`/`next` theo nhiều kiểu) và đối chiếu với một lời giải độc lập mô phỏng thứ tự "xen kẽ theo cột".

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề **thiết kế iterator (bộ lặp)**: cho hai vector số nguyên `v1`, `v2`, cài đặt lớp `ZigzagIterator` trả về các phần tử **xen kẽ** (`v1[0], v2[0], v1[1], v2[1], ...`), **bỏ qua** vector đã hết khi vector kia còn phần tử; có `hasNext()` và `next()`.
2. Hiểu **hợp đồng của iterator**: `hasNext()` **không làm tiêu hao** phần tử (gọi nhiều lần cho cùng kết quả); `next()` trả phần tử kế và tiến lên; và nên làm cho `next()` **đúng cả khi người dùng không gọi `hasNext()` trước**.
3. Đi từ **trải phẳng ngay khi khởi tạo** → **hai con trỏ + biến lượt (cách tham khảo)** → **hàng đợi các vector còn phần tử** (tổng quát cho `k` vector, trả lời follow up) → **bộ sinh lười `zip_longest` có xem trước một phần tử** → **tổng quát hóa cho `k` vector**.
4. Nắm khái niệm **iterator, sinh lười (lazy), xem trước (lookahead)**, và hai thao tác `deque` (`popleft`, `append`) để quay vòng.
5. Tránh các lỗi kinh điển: không bỏ qua vector đã hết (`IndexError`), `hasNext` chỉ kiểm tra vector hiện tại (trả `False` sớm), sửa mảng đầu vào bằng `pop(0)`, và dùng `zip` làm mất phần dư của vector dài hơn.

## 2. Đề bài & ví dụ

Cho hai vector số nguyên `v1` và `v2`, hãy cài đặt một **iterator** trả về các phần tử của chúng **xen kẽ nhau** (lần lượt lấy từ `v1`, rồi `v2`, rồi `v1`, ...). Khi một vector hết, tiếp tục lấy từ vector còn lại.

Cài đặt lớp `ZigzagIterator`:

- `ZigzagIterator(List<int> v1, List<int> v2)`: khởi tạo với hai vector `v1`, `v2`;
- `boolean hasNext()`: trả `true` nếu iterator còn phần tử, ngược lại `false`;
- `int next()`: trả về phần tử hiện tại và chuyển sang phần tử kế tiếp.

**Ví dụ 1**

- Input: `v1 = [1,2]`, `v2 = [3,4,5,6]`
- Output: `[1,3,2,4,5,6]`
- Giải thích: gọi `next` liên tiếp cho tới khi `hasNext` trả `false`, thứ tự phần tử là `[1,3,2,4,5,6]`.

**Ví dụ 2**

- Input: `v1 = [1]`, `v2 = []`
- Output: `[1]`

**Ví dụ 3**

- Input: `v1 = []`, `v2 = [1]`
- Output: `[1]`

**Ràng buộc:** `0 <= v1.length, v2.length <= 1000`; `1 <= v1.length + v2.length <= 2000`; `-2³¹ <= v1[i], v2[i] <= 2³¹ - 1`.

**Follow up:** nếu cho `k` vector thì mã của bạn mở rộng được đến đâu? (Với `k > 2` thứ tự "zigzag" không rõ ràng; có thể thay bằng "xoay vòng": ví dụ `v1 = [1,2,3]`, `v2 = [4,5,6,7]`, `v3 = [8,9]` cho `[1,4,8,2,5,9,3,6,7]`.)

In [1]:
from typing import List
import random
from collections import deque
from itertools import zip_longest

examples = [
    ([[1, 2], [3, 4, 5, 6]], [1, 3, 2, 4, 5, 6]),
    ([[1], []], [1]),
    ([[], [1]], [1]),
    ([[1, 2, 3], [4, 5, 6, 7], [8, 9]], [1, 4, 8, 2, 5, 9, 3, 6, 7]),       # follow up, k = 3
]
for vectors, expected in examples:
    print(f"vectors={vectors}, kỳ vọng {expected}")

vectors=[[1, 2], [3, 4, 5, 6]], kỳ vọng [1, 3, 2, 4, 5, 6]
vectors=[[1], []], kỳ vọng [1]
vectors=[[], [1]], kỳ vọng [1]
vectors=[[1, 2, 3], [4, 5, 6, 7], [8, 9]], kỳ vọng [1, 4, 8, 2, 5, 9, 3, 6, 7]


## 3. Kiến thức nền

### 3.1. Iterator là gì? `hasNext` và `next`

Iterator là đối tượng cho phép **lấy từng phần tử một**: `hasNext()` cho biết còn phần tử không, `next()` trả phần tử kế và **tiến lên**. Cách dùng chuẩn: `while it.hasNext(): value = it.next()`. (Python có giao thức riêng với `__iter__`/`__next__` và `StopIteration`; bài này dùng cặp phương thức kiểu Java.) Điều quan trọng: `hasNext()` **không được thay đổi** dãy phần tử còn lại: gọi hai lần liên tiếp phải cho cùng kết quả.

In [2]:
class RangeIterator:
    """Ví dụ nhỏ: iterator duyệt qua các số 0..n-1."""

    def __init__(self, n):
        self.i, self.n = 0, n

    def hasNext(self):
        return self.i < self.n

    def next(self):
        value = self.i
        self.i += 1
        return value


it = RangeIterator(3)
out = []
while it.hasNext():
    out.append(it.next())
print(out)                           # [0, 1, 2]
print(it.hasNext(), it.hasNext())    # False False (gọi nhiều lần vẫn như nhau)

[0, 1, 2]
False False


### 3.2. Thuộc tính đối tượng: giữ trạng thái giữa các lần gọi

Vì `next()` được gọi nhiều lần, lớp phải **nhớ** mình đang đứng ở đâu: ta dùng thuộc tính của đối tượng (`self.xxx`) lưu **chỉ số hiện tại** của mỗi vector và **vector đang tới lượt**. Mỗi đối tượng có bộ thuộc tính riêng.

In [3]:
class TwoCounters:
    def __init__(self):
        self.pos = [0, 0]
        self.cur = 0

    def step(self):
        self.pos[self.cur] += 1
        self.cur = (self.cur + 1) % 2        # quay vòng 0 -> 1 -> 0 -> ...
        return self.pos[:], self.cur


tc = TwoCounters()
print(tc.step(), tc.step(), tc.step())

([1, 0], 1) ([1, 1], 0) ([2, 1], 1)


### 3.3. Quay vòng bằng toán tử `%`

`(cur + 1) % k` đưa chỉ số từ `0` tới `k − 1` rồi **quay về `0`**. Đây là cách luân phiên giữa `k` vector (`k = 2` cho zigzag, `k` bất kỳ cho "xoay vòng").

In [4]:
k = 3
cur = 0
order = []
for _ in range(7):
    order.append(cur)
    cur = (cur + 1) % k
print(order)                                # [0, 1, 2, 0, 1, 2, 0]

[0, 1, 2, 0, 1, 2, 0]


### 3.4. `collections.deque` làm hàng đợi quay vòng

`deque` cho `popleft()` (lấy phần tử đầu) và `append()` (thêm vào cuối) đều `O(1)`. Dùng hàng đợi các "vector còn phần tử": mỗi lần lấy vector ở đầu, lấy một phần tử, rồi nếu vector còn phần tử thì **đưa lại vào cuối**; vector đã hết thì **không đưa lại**. Cách này tự động bỏ qua vector hết và quay vòng đúng thứ tự.

In [5]:
q = deque(["A", "B", "C"])
for _ in range(5):
    x = q.popleft()
    print("lấy", x, end="; ")
    if x != "B":                       # giả sử B đã hết nên không đưa lại
        q.append(x)
print()
print(list(q))

lấy A; lấy B; lấy C; lấy A; lấy C; 
['A', 'C']


### 3.5. `itertools.zip_longest` và bộ sinh (generator) lười

`zip_longest(a, b, fillvalue=X)` ghép phần tử cùng vị trí và **điền `X`** khi một dãy hết (khác `zip` dừng ở dãy ngắn nhất). Ghép `zip_longest(v1, v2)` rồi trải phẳng bằng biểu thức sinh (generator expression) cho ta thứ tự xen kẽ **lười** (chỉ tính khi cần). Một iterator Python chỉ cho `next(gen)` **một lần** mỗi phần tử, nên để cài `hasNext()` ta phải **xem trước (prefetch)** một phần tử và nhớ nó.

In [6]:
sentinel = object()
v1, v2 = [1, 2], [3, 4, 5, 6]
gen = (x for pair in zip_longest(v1, v2, fillvalue=sentinel) for x in pair if x is not sentinel)
print(list(gen))                               # [1, 3, 2, 4, 5, 6]
print(list(zip([1, 2], [3, 4, 5, 6])))         # [(1, 3), (2, 4)]  zip cắt bỏ 5, 6

[1, 3, 2, 4, 5, 6]
[(1, 3), (2, 4)]


### 3.6. Dựng khung test cho bài thiết kế iterator

Ta viết hàm `drain(iterator)` lấy hết phần tử bằng `while hasNext(): next()`, và nhiều kiểu gọi khác (gọi `hasNext` lặp lại, gọi `next` không cần `hasNext`...) để kiểm tra mọi lớp **đúng hợp đồng**.

In [7]:
def drain(it):
    out = []
    while it.hasNext():
        out.append(it.next())
    return out


print(drain(RangeIterator(4)))             # [0, 1, 2, 3]

[0, 1, 2, 3]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

`v1 = [1, 2]`, `v2 = [3, 4, 5, 6]`. Xen kẽ lần lượt từ `v1` rồi `v2`:

| lần `next` | lượt của | phần tử lấy | `v1` còn | `v2` còn |
|---|---|---|---|---|
| 1 | `v1` | `1` | `[2]` | `[3,4,5,6]` |
| 2 | `v2` | `3` | `[2]` | `[4,5,6]` |
| 3 | `v1` | `2` | `[]` | `[4,5,6]` |
| 4 | `v2` | `4` | `[]` | `[5,6]` |
| 5 | `v1` hết → bỏ qua, sang `v2` | `5` | `[]` | `[6]` |
| 6 | `v2` | `6` | `[]` | `[]` |

Kết quả `[1, 3, 2, 4, 5, 6]`. Điều đáng chú ý ở lần `5`: tới lượt `v1` nhưng `v1` đã hết nên **phải nhảy sang `v2`**. Cũng cần chú ý trường hợp vector đầu rỗng (`v1 = []`, `v2 = [1]`): lượt đầu tiên là `v1` nhưng nó rỗng.

Ta cần một cơ chế để **luôn đứng ở vector còn phần tử** mỗi khi hỏi `hasNext` hoặc lấy `next`.

In [8]:
v1, v2 = [1, 2], [3, 4, 5, 6]
vectors, idx, cur = [v1, v2], [0, 0], 0
out = []
while any(idx[j] < len(vectors[j]) for j in range(2)):
    while idx[cur] >= len(vectors[cur]):            # vector tới lượt đã hết: chuyển sang vector kế
        cur = (cur + 1) % 2
    out.append(vectors[cur][idx[cur]])
    idx[cur] += 1
    cur = (cur + 1) % 2
print(out)

[1, 3, 2, 4, 5, 6]


## 5. Cách 1 — Brute Force / baseline: trải phẳng ngay khi khởi tạo

**Ý tưởng.** Trong `__init__`, dựng sẵn **toàn bộ dãy xen kẽ** vào một list `flat`; giữ con trỏ `pos`. `hasNext` là `pos < len(flat)`, `next` trả `flat[pos]` rồi `pos += 1`. Đơn giản và đúng nhưng tốn bộ nhớ `O(n)` và làm việc ngay cả khi người dùng không đọc hết.

In [9]:
class ZigzagFlatten:
    def __init__(self, v1: list[int], v2: list[int]):
        self.flat = []
        for i in range(max(len(v1), len(v2))):         # duyệt theo cột: cột i gồm v1[i] rồi v2[i]
            if i < len(v1):
                self.flat.append(v1[i])
            if i < len(v2):
                self.flat.append(v2[i])
        self.pos = 0

    def next(self) -> int:
        value = self.flat[self.pos]
        self.pos += 1
        return value

    def hasNext(self) -> bool:
        return self.pos < len(self.flat)


for vectors, expected in examples[:3]:
    print(f"{vectors} -> {drain(ZigzagFlatten(*vectors))} (kỳ vọng {expected})")

[[1, 2], [3, 4, 5, 6]] -> [1, 3, 2, 4, 5, 6] (kỳ vọng [1, 3, 2, 4, 5, 6])
[[1], []] -> [1] (kỳ vọng [1])
[[], [1]] -> [1] (kỳ vọng [1])


**Độ phức tạp (Cách 1).** `n = len(v1) + len(v2)`: khởi tạo **`O(n)`** thời gian và **`O(n)`** bộ nhớ (list `flat`); `next` và `hasNext` **`O(1)`**.

**Nút thắt.** Ta sao chép mọi phần tử ngay từ đầu dù chưa cần. Một iterator đúng nghĩa nên chỉ giữ **vị trí** (và tham chiếu tới dữ liệu gốc) rồi tính phần tử kế khi được hỏi.

## 6. Ý tưởng thuật toán / bước đột phá

**Giữ vị trí thay vì sao chép dữ liệu.** Với `k` vector, lưu `indexes[j]` là vị trí kế tiếp cần đọc của vector `j` và `cur` là vector tới lượt. Hai thao tác:

- `hasNext()`: **quay `cur` tới vector còn phần tử** (`indexes[cur] < len(vectors[cur])`); nếu quay một vòng đầy đủ mà không có vector nào còn phần tử thì trả `False`, ngược lại `True`. Thao tác này **không đổi dãy phần tử còn lại** (chỉ dời `cur` qua các vector đã hết, điều có thể làm lặp lại mà không hại gì).
- `next()`: gọi `hasNext()` để chắc `cur` trỏ vector còn phần tử, lấy `vectors[cur][indexes[cur]]`, tăng `indexes[cur]`, rồi chuyển `cur` sang vector kế `(cur + 1) % k`.

Việc `next()` tự gọi `hasNext()` làm cho lớp **an toàn** khi người dùng không gọi `hasNext()` trước (mà vẫn còn phần tử). Mỗi thao tác `O(k)` trường hợp xấu (duyệt qua các vector hết), `O(1)` khi `k = 2`.

Một cách khác tự nhiên hơn: **hàng đợi các vector còn phần tử**: chỉ chứa những vector chưa hết; mỗi `next` lấy vector đầu hàng, đọc một phần tử, và đưa lại vào cuối nếu còn. Khi đó mọi thao tác đúng `O(1)` và không cần dời `cur`.

In [10]:
vectors = [[1, 2], [3, 4, 5, 6]]
indexes, cur = [0, 0], 0
k = len(vectors)


def has_next():
    global cur
    start = cur
    while indexes[cur] == len(vectors[cur]):
        cur = (cur + 1) % k
        if cur == start:
            return False
    return True


def take():
    global cur
    assert has_next()
    value = vectors[cur][indexes[cur]]
    indexes[cur] += 1
    cur = (cur + 1) % k
    return value


out = []
while has_next():
    out.append(take())
print(out)

[1, 3, 2, 4, 5, 6]


## 7. Các cách tiếp theo

### Cách 2 — Hai con trỏ + biến lượt (cách của lời giải tham khảo)

**Giải thích.** `vectors = [v1, v2]`, `indexes = [0, 0]`, `cur = 0`. `hasNext()` quay `cur` tới vector còn phần tử (dừng khi đã quay một vòng mà không có). `next()` đọc `vectors[cur][indexes[cur]]`, tăng chỉ số, đổi `cur`. Phiên bản dưới đây bổ sung: `next()` gọi `hasNext()` trước để an toàn.

In [11]:
class ZigzagPointers:
    def __init__(self, v1: list[int], v2: list[int]):
        self.vectors = [v1, v2]
        self.size = 2
        self.indexes = [0] * self.size
        self.cur = 0

    def next(self) -> int:
        self.hasNext()                                       # bảo đảm cur trỏ vector còn phần tử
        vector = self.vectors[self.cur]
        index = self.indexes[self.cur]
        value = vector[index]
        self.indexes[self.cur] = index + 1
        self.cur = (self.cur + 1) % self.size                # chuyển lượt
        return value

    def hasNext(self) -> bool:
        start = self.cur
        while self.indexes[self.cur] == len(self.vectors[self.cur]):
            self.cur = (self.cur + 1) % self.size            # bỏ qua vector đã hết
            if self.cur == start:
                return False                                  # đã quay đủ vòng: không còn gì
        return True


for vectors, expected in examples[:3]:
    print(f"{vectors} -> {drain(ZigzagPointers(*vectors))} (kỳ vọng {expected})")

[[1, 2], [3, 4, 5, 6]] -> [1, 3, 2, 4, 5, 6] (kỳ vọng [1, 3, 2, 4, 5, 6])
[[1], []] -> [1] (kỳ vọng [1])
[[], [1]] -> [1] (kỳ vọng [1])


In [12]:
# Code nộp LeetCode cho Cách 2
from typing import List


class ZigzagIterator:
    def __init__(self, v1: List[int], v2: List[int]):
        self.cur = 0
        self.size = 2
        self.indexes = [0] * self.size
        self.vectors = [v1, v2]

    def next(self) -> int:
        self.hasNext()
        vector = self.vectors[self.cur]
        index = self.indexes[self.cur]
        res = vector[index]
        self.indexes[self.cur] = index + 1
        self.cur = (self.cur + 1) % self.size
        return res

    def hasNext(self) -> bool:
        start = self.cur
        while self.indexes[self.cur] == len(self.vectors[self.cur]):
            self.cur = (self.cur + 1) % self.size
            if self.cur == start:
                return False
        return True

**Độ phức tạp Cách 2:** khởi tạo **`O(1)`**, mỗi thao tác **`O(k)`** (`k = 2` nên `O(1)`), **Space `O(k)`** thêm (không sao chép dữ liệu; chỉ giữ tham chiếu tới các vector).

### Cách 3 — Hàng đợi các vector còn phần tử (`O(1)` mỗi thao tác, mở rộng cho `k` vector)

**Giải thích.** `queue` là `deque` các cặp `(vector, chỉ_số)` của những vector **còn phần tử**. `next()`: `vector, i = queue.popleft()`; `value = vector[i]`; nếu `i + 1 < len(vector)` thì `queue.append((vector, i + 1))`; trả `value`. `hasNext()`: `bool(queue)`. Vector rỗng không được đưa vào hàng đợi ngay từ đầu.

In [13]:
class ZigzagQueue:
    def __init__(self, v1: list[int], v2: list[int], *more: list[int]):
        self.queue = deque((v, 0) for v in (v1, v2, *more) if v)       # chỉ các vector không rỗng

    def next(self) -> int:
        vector, i = self.queue.popleft()                                # vector tới lượt
        if i + 1 < len(vector):
            self.queue.append((vector, i + 1))                           # còn phần tử: quay lại cuối hàng
        return vector[i]

    def hasNext(self) -> bool:
        return bool(self.queue)


for vectors, expected in examples:
    print(f"{vectors} -> {drain(ZigzagQueue(*vectors))} (kỳ vọng {expected})")

[[1, 2], [3, 4, 5, 6]] -> [1, 3, 2, 4, 5, 6] (kỳ vọng [1, 3, 2, 4, 5, 6])
[[1], []] -> [1] (kỳ vọng [1])
[[], [1]] -> [1] (kỳ vọng [1])
[[1, 2, 3], [4, 5, 6, 7], [8, 9]] -> [1, 4, 8, 2, 5, 9, 3, 6, 7] (kỳ vọng [1, 4, 8, 2, 5, 9, 3, 6, 7])


**Độ phức tạp Cách 3:** khởi tạo `O(k)`, `next` và `hasNext` đều **`O(1)`**, **Space `O(k)`**. Mở rộng tự nhiên cho `k` vector (câu trả lời cho follow up): chỉ cần thêm vector vào `deque` ban đầu.

### Cách 4 — Bộ sinh lười `zip_longest` có xem trước một phần tử

**Giải thích.** Tạo generator xen kẽ bằng `zip_longest(*vectors, fillvalue=SENTINEL)` rồi bỏ `SENTINEL`. Vì `next(gen)` tiêu hao phần tử, để cài `hasNext()` ta **xem trước** một phần tử và lưu vào `self.buffer`; `next()` trả bộ đệm rồi xóa nó. Không sao chép dữ liệu, tính lười.

In [14]:
class ZigzagLazy:
    _SENTINEL = object()

    def __init__(self, v1: list[int], v2: list[int], *more: list[int]):
        sentinel = self._SENTINEL
        self.gen = (x for row in zip_longest(v1, v2, *more, fillvalue=sentinel) for x in row if x is not sentinel)
        self.has_buffer = False
        self.buffer = None

    def _fill(self):
        if not self.has_buffer:
            try:
                self.buffer = next(self.gen)                 # xem trước một phần tử
                self.has_buffer = True
            except StopIteration:
                pass

    def hasNext(self) -> bool:
        self._fill()
        return self.has_buffer

    def next(self) -> int:
        self._fill()
        value = self.buffer
        self.has_buffer = False                               # đã tiêu thụ bộ đệm
        self.buffer = None
        return value


for vectors, expected in examples:
    print(f"{vectors} -> {drain(ZigzagLazy(*vectors))} (kỳ vọng {expected})")

[[1, 2], [3, 4, 5, 6]] -> [1, 3, 2, 4, 5, 6] (kỳ vọng [1, 3, 2, 4, 5, 6])
[[1], []] -> [1] (kỳ vọng [1])
[[], [1]] -> [1] (kỳ vọng [1])
[[1, 2, 3], [4, 5, 6, 7], [8, 9]] -> [1, 4, 8, 2, 5, 9, 3, 6, 7] (kỳ vọng [1, 4, 8, 2, 5, 9, 3, 6, 7])


**Độ phức tạp Cách 4:** khởi tạo `O(1)`, mỗi thao tác **`O(1)` khấu hao** (`zip_longest` đã làm việc theo cột; việc bỏ sentinel tốn thêm tối đa `k` mỗi cột), **Space `O(k)`**.

### Cách 5 — Mở rộng: lớp tổng quát cho `k` vector bằng con trỏ và biến lượt

**Giải thích.** Chỉ cần tổng quát hóa Cách 2: nhận danh sách `vectors` bất kỳ (`k` vector), `indexes = [0] * k`, và quay `cur` theo `% k`. Dùng cùng `hasNext`/`next`. Đây là mở rộng cho follow up bằng **con trỏ**; Cách 3 là mở rộng bằng **hàng đợi**.

In [15]:
class ZigzagIteratorK:
    def __init__(self, *vectors: list[int]):
        self.vectors = list(vectors)
        self.size = len(self.vectors)
        self.indexes = [0] * self.size
        self.cur = 0

    def hasNext(self) -> bool:
        if self.size == 0:
            return False
        start = self.cur
        while self.indexes[self.cur] == len(self.vectors[self.cur]):
            self.cur = (self.cur + 1) % self.size
            if self.cur == start:
                return False
        return True

    def next(self) -> int:
        self.hasNext()
        value = self.vectors[self.cur][self.indexes[self.cur]]
        self.indexes[self.cur] += 1
        self.cur = (self.cur + 1) % self.size
        return value


for vectors, expected in examples:
    print(f"{vectors} -> {drain(ZigzagIteratorK(*vectors))} (kỳ vọng {expected})")

[[1, 2], [3, 4, 5, 6]] -> [1, 3, 2, 4, 5, 6] (kỳ vọng [1, 3, 2, 4, 5, 6])
[[1], []] -> [1] (kỳ vọng [1])
[[], [1]] -> [1] (kỳ vọng [1])
[[1, 2, 3], [4, 5, 6, 7], [8, 9]] -> [1, 4, 8, 2, 5, 9, 3, 6, 7] (kỳ vọng [1, 4, 8, 2, 5, 9, 3, 6, 7])


**Độ phức tạp Cách 5:** khởi tạo `O(k)`, mỗi thao tác `O(k)` trong trường hợp xấu (dời `cur` qua nhiều vector hết), **Space `O(k)`**.

> Cách 2 là cách nộp bài. Cách 3 là cách tốt nhất khi `k` lớn. Cách 4 minh họa phong cách lười của Python. Cách 1 chỉ để đối chiếu.

## 8. TRACE từng bước

Ta theo dõi Cách 2 (hai con trỏ) trên `v1 = [1,2]`, `v2 = [3,4,5,6]` (in `cur`, `indexes`) và Cách 3 (hàng đợi) trên `v1 = [1,2,3]`, `v2 = [4,5,6,7]`, `v3 = [8,9]`.

In [16]:
def trace_pointers(v1, v2):
    it = ZigzagPointers(v1, v2)
    print(f"v1={v1}, v2={v2}")
    step = 0
    while it.hasNext():
        step += 1
        before = (it.cur, list(it.indexes))
        value = it.next()
        print(f"  next #{step}: cur={before[0]}, indexes={before[1]} -> trả {value}; cur mới = {it.cur}, indexes = {it.indexes}")
    print("  hasNext() = False: kết thúc")


trace_pointers([1, 2], [3, 4, 5, 6])

v1=[1, 2], v2=[3, 4, 5, 6]
  next #1: cur=0, indexes=[0, 0] -> trả 1; cur mới = 1, indexes = [1, 0]
  next #2: cur=1, indexes=[1, 0] -> trả 3; cur mới = 0, indexes = [1, 1]
  next #3: cur=0, indexes=[1, 1] -> trả 2; cur mới = 1, indexes = [2, 1]
  next #4: cur=1, indexes=[2, 1] -> trả 4; cur mới = 0, indexes = [2, 2]
  next #5: cur=1, indexes=[2, 2] -> trả 5; cur mới = 0, indexes = [2, 3]
  next #6: cur=1, indexes=[2, 3] -> trả 6; cur mới = 0, indexes = [2, 4]
  hasNext() = False: kết thúc


In [17]:
def trace_queue(*vectors):
    it = ZigzagQueue(*vectors)
    print(f"vectors={list(vectors)}; hàng đợi ban đầu (vector, chỉ số) = {[(v, i) for v, i in it.queue]}")
    out = []
    while it.hasNext():
        head = it.queue[0]
        value = it.next()
        out.append(value)
        print(f"  lấy {value} từ {head[0]} (chỉ số {head[1]}) -> hàng đợi còn {[(v, i) for v, i in it.queue]}")
    print("  =>", out)
    return out


assert trace_queue([1, 2, 3], [4, 5, 6, 7], [8, 9]) == [1, 4, 8, 2, 5, 9, 3, 6, 7]

vectors=[[1, 2, 3], [4, 5, 6, 7], [8, 9]]; hàng đợi ban đầu (vector, chỉ số) = [([1, 2, 3], 0), ([4, 5, 6, 7], 0), ([8, 9], 0)]
  lấy 1 từ [1, 2, 3] (chỉ số 0) -> hàng đợi còn [([4, 5, 6, 7], 0), ([8, 9], 0), ([1, 2, 3], 1)]
  lấy 4 từ [4, 5, 6, 7] (chỉ số 0) -> hàng đợi còn [([8, 9], 0), ([1, 2, 3], 1), ([4, 5, 6, 7], 1)]
  lấy 8 từ [8, 9] (chỉ số 0) -> hàng đợi còn [([1, 2, 3], 1), ([4, 5, 6, 7], 1), ([8, 9], 1)]
  lấy 2 từ [1, 2, 3] (chỉ số 1) -> hàng đợi còn [([4, 5, 6, 7], 1), ([8, 9], 1), ([1, 2, 3], 2)]
  lấy 5 từ [4, 5, 6, 7] (chỉ số 1) -> hàng đợi còn [([8, 9], 1), ([1, 2, 3], 2), ([4, 5, 6, 7], 2)]
  lấy 9 từ [8, 9] (chỉ số 1) -> hàng đợi còn [([1, 2, 3], 2), ([4, 5, 6, 7], 2)]
  lấy 3 từ [1, 2, 3] (chỉ số 2) -> hàng đợi còn [([4, 5, 6, 7], 2)]
  lấy 6 từ [4, 5, 6, 7] (chỉ số 2) -> hàng đợi còn [([4, 5, 6, 7], 3)]
  lấy 7 từ [4, 5, 6, 7] (chỉ số 3) -> hàng đợi còn []
  => [1, 4, 8, 2, 5, 9, 3, 6, 7]


Bảng tóm tắt (Cách 2, `v1 = [1,2]`, `v2 = [3,4,5,6]`):

| lần | `cur` trước | `indexes` trước | trả về | `cur` sau |
|---|---|---|---|---|
| 1 | 0 | `[0, 0]` | `1` | 1 |
| 2 | 1 | `[1, 0]` | `3` | 0 |
| 3 | 0 | `[1, 1]` | `2` | 1 |
| 4 | 1 | `[2, 1]` | `4` | 0 |
| 5 | 0 (`indexes[0] = 2 = len(v1)`: hết) → `hasNext` dời `cur` sang 1 | `[2, 2]` | `5` | 0 |
| 6 | 0 (hết) → dời sang 1 | `[2, 3]` | `6` | 0 |

Sau lần 6 cả hai vector hết, `hasNext()` quay đủ một vòng và trả `False`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (hai con trỏ + biến lượt; `next()` tự đảm bảo trạng thái bằng cách gọi `hasNext()`). Đúng tên lớp `ZigzagIterator`, đúng các phương thức `__init__(self, v1, v2)`, `next(self)`, `hasNext(self)`.

In [18]:
from typing import List


class ZigzagIterator:
    def __init__(self, v1: List[int], v2: List[int]):
        self.cur = 0                                   # chỉ số của vector tới lượt
        self.size = 2
        self.indexes = [0] * self.size                 # vị trí kế tiếp cần đọc của từng vector
        self.vectors = [v1, v2]

    def next(self) -> int:
        self.hasNext()                                 # bảo đảm self.cur trỏ vector còn phần tử
        vector = self.vectors[self.cur]
        index = self.indexes[self.cur]
        res = vector[index]
        self.indexes[self.cur] = index + 1
        self.cur = (self.cur + 1) % self.size          # lượt kế tiếp
        return res

    def hasNext(self) -> bool:
        start = self.cur
        while self.indexes[self.cur] == len(self.vectors[self.cur]):
            self.cur = (self.cur + 1) % self.size      # bỏ qua vector đã hết
            if self.cur == start:
                return False                           # quay đủ một vòng: hết cả hai
        return True


for vectors, expected in examples[:3]:
    i, v = ZigzagIterator(*vectors), []
    while i.hasNext():
        v.append(i.next())
    assert v == expected
print("PASS ví dụ đề cho class ZigzagIterator")

PASS ví dụ đề cho class ZigzagIterator


## 10. Vì sao thuật toán đúng

- **Thứ tự xen kẽ đúng định nghĩa:** `cur` luân phiên `0, 1, 0, 1, ...` sau mỗi `next`, và khi tới lượt một vector đã hết, `hasNext` dời `cur` sang vector kế. Với `k = 2`, thứ tự là `v1[0], v2[0], v1[1], v2[1], ...` cho tới khi một vector hết; sau đó chỉ còn vector kia nên lần lượt các phần tử còn lại. Đây chính xác là thứ tự "theo cột" `(v1[i], v2[i])` với cột `i` bỏ phần tử nào không tồn tại.
- **`hasNext` đúng và không tiêu hao:** `hasNext` chỉ thay đổi `cur` (không thay đổi `indexes`), nên số phần tử còn lại giữ nguyên. Nó dừng ở vector đầu tiên còn phần tử (trả `True`) hoặc sau một vòng đầy đủ không thấy (trả `False`). Vì mọi vector hết đồng nghĩa không còn phần tử nào.
- **`next` an toàn:** `next` gọi `hasNext` nên `cur` luôn trỏ vector còn phần tử; truy cập `vector[index]` hợp lệ khi còn phần tử (nếu không còn, hành vi không xác định theo đề: "gọi `next` hợp lệ").
- **Hàng đợi (Cách 3):** hàng đợi luôn chứa đúng các vector còn phần tử theo thứ tự lượt; lấy đầu, đọc một phần tử, đưa lại vào cuối nếu còn: đúng là xoay vòng và tự bỏ qua vector hết.
- **Cách 4:** `zip_longest` cho các cột theo thứ tự; loại sentinel bỏ các vị trí không tồn tại; bộ đệm một phần tử cho `hasNext` không tiêu hao.
- **Cách 1:** dựng toàn bộ dãy bằng cùng quy tắc theo cột.

## 11. Phân tích Time / Space Complexity

Gọi `n` là tổng số phần tử, `k` là số vector (`k = 2` trong đề).

- **Cách 1 (trải phẳng):** khởi tạo `O(n)`, mỗi thao tác `O(1)`; Space `O(n)`.
- **Cách 2 (hai con trỏ):** khởi tạo `O(1)`, mỗi thao tác `O(k)` (`O(1)` khi `k = 2`); Space `O(k)`.
- **Cách 3 (hàng đợi):** khởi tạo `O(k)`, mỗi thao tác `O(1)`; Space `O(k)`.
- **Cách 4 (`zip_longest` + xem trước):** khởi tạo `O(1)`, mỗi thao tác `O(1)` khấu hao (tối đa `O(k)` khi gặp nhiều sentinel liên tiếp trong một hàng); Space `O(k)`.
- **Cách 5 (tổng quát `k` vector bằng con trỏ):** mỗi thao tác `O(k)`; Space `O(k)`.
- Duyệt hết toàn bộ phần tử mất `O(n)` (hoặc `O(n·k)` cho Cách 2/5 trong trường hợp xấu). Dữ liệu gốc **không bị sao chép hay sửa** ở các cách 2–5.

## 12. Bộ test

`oracle(vectors)` mô phỏng thứ tự **xoay vòng theo cột** bằng hai vòng lặp trên chỉ số (độc lập với các lớp). Ngoài việc lấy hết bằng `while hasNext: next`, ta kiểm tra thêm: **gọi `hasNext()` lặp lại** không đổi kết quả, **gọi `next()` không cần `hasNext()`** (miễn còn phần tử), và **dữ liệu đầu vào không bị sửa**.

In [19]:
sol_cls = ZigzagIterator
impls_two = {
    "flatten": ZigzagFlatten,
    "pointers": ZigzagPointers,
    "queue": ZigzagQueue,
    "lazy": ZigzagLazy,
    "k_pointers": ZigzagIteratorK,
    "submit": sol_cls,
}
impls_k = {"queue": ZigzagQueue, "lazy": ZigzagLazy, "k_pointers": ZigzagIteratorK}


def oracle(vectors):
    result = []
    i = 0
    while any(i < len(v) for v in vectors):
        for v in vectors:
            if i < len(v):
                result.append(v[i])
        i += 1
    return result


def drain_repeat_hasnext(it, repeats=3):
    out = []
    while True:
        answers = [it.hasNext() for _ in range(repeats)]          # gọi nhiều lần: phải nhất quán
        assert len(set(answers)) == 1, answers
        if not answers[0]:
            break
        out.append(it.next())
    return out


def drain_next_only(it, count):
    """Gọi next() đúng count lần, không gọi hasNext() trước."""
    return [it.next() for _ in range(count)]


def run_two(cls, v1, v2):
    a, b = list(v1), list(v2)
    res = (drain(cls(a, b)), drain_repeat_hasnext(cls(list(v1), list(v2))), drain_next_only(cls(list(v1), list(v2)), len(v1) + len(v2)))
    assert a == v1 and b == v2, (cls.__name__, "đã sửa đầu vào")
    return res


for vectors, expected in examples[:3]:
    assert oracle(vectors) == expected
    for name, cls in impls_two.items():
        r1, r2, r3 = run_two(cls, *vectors)
        assert r1 == r2 == r3 == expected, (name, vectors, r1, r2, r3, expected)
for vectors, expected in examples[3:]:
    assert oracle(vectors) == expected
    for name, cls in impls_k.items():
        assert drain(cls(*[list(v) for v in vectors])) == expected, (name, vectors)
print("PASS ví dụ của đề và follow up (mọi lớp, mọi kiểu gọi; oracle độc lập)")

PASS ví dụ của đề và follow up (mọi lớp, mọi kiểu gọi; oracle độc lập)


### 12.1. Các trường hợp biên

In [20]:
def check_two(v1, v2, label):
    expected = oracle([v1, v2])
    for name, cls in impls_two.items():
        r1, r2, r3 = run_two(cls, v1, v2)
        assert r1 == r2 == r3 == expected, (label, name, r1, r2, r3, expected)
    print("PASS", label, "->", expected if len(expected) <= 8 else f"{len(expected)} phần tử")


check_two([1], [], "v2 rỗng")
check_two([], [1], "v1 rỗng (lượt đầu là vector rỗng)")
check_two([1], [2], "mỗi vector một phần tử")
check_two([1, 2], [3, 4, 5, 6], "ví dụ 1 của đề")
check_two([1, 2, 3, 4], [5], "v1 dài hơn")
check_two([1], [2, 3, 4, 5], "v2 dài hơn")
check_two([1, 2, 3], [4, 5, 6], "hai vector cùng độ dài")
check_two([-2**31, 2**31 - 1], [0], "giá trị biên 32 bit")
check_two(list(range(1000)), list(range(1000, 2000)), "n = 2000, cùng độ dài 1000")
check_two(list(range(1000)), [], "n = 1000, một vector rỗng")
check_two([], list(range(1000)), "n = 1000, vector đầu rỗng")

PASS v2 rỗng -> [1]
PASS v1 rỗng (lượt đầu là vector rỗng) -> [1]
PASS mỗi vector một phần tử -> [1, 2]
PASS ví dụ 1 của đề -> [1, 3, 2, 4, 5, 6]
PASS v1 dài hơn -> [1, 5, 2, 3, 4]
PASS v2 dài hơn -> [1, 2, 3, 4, 5]
PASS hai vector cùng độ dài -> [1, 4, 2, 5, 3, 6]
PASS giá trị biên 32 bit -> [-2147483648, 0, 2147483647]
PASS n = 2000, cùng độ dài 1000 -> 2000 phần tử
PASS n = 1000, một vector rỗng -> 1000 phần tử
PASS n = 1000, vector đầu rỗng -> 1000 phần tử


### 12.2. Tính chất và phản ví dụ

In [21]:
random.seed(281)

# Tính chất 1: kết quả là hoán vị của hợp hai vector; thứ tự tương đối trong từng vector được bảo toàn
for _ in range(300):
    v1 = [random.randint(-9, 9) for _ in range(random.randint(0, 12))]
    v2 = [random.randint(-9, 9) for _ in range(random.randint(0, 12))]
    if not v1 and not v2:
        continue
    out = drain(ZigzagIterator(list(v1), list(v2)))
    assert sorted(out) == sorted(v1 + v2)
    # tách lại: phần tử ở vị trí xen kẽ ban đầu thuộc v1 / v2 theo quy tắc cột
    assert out == oracle([v1, v2])
print("PASS tính chất: là hoán vị của hợp hai vector và đúng thứ tự xen kẽ")

# Tính chất 2: nhiều vòng gọi hasNext/next xen kẽ ngẫu nhiên vẫn đúng
for _ in range(300):
    v1 = [random.randint(0, 9) for _ in range(random.randint(0, 10))]
    v2 = [random.randint(0, 9) for _ in range(random.randint(0, 10))]
    if not v1 and not v2:
        continue
    for name, cls in impls_two.items():
        it = cls(list(v1), list(v2))
        out = []
        while len(out) < len(v1) + len(v2):
            if random.random() < 0.5:
                assert it.hasNext() is True
            out.append(it.next())
            for _ in range(random.randint(0, 2)):
                it.hasNext()
        assert out == oracle([v1, v2]) and it.hasNext() is False, (name, v1, v2)
print("PASS tính chất: gọi hasNext tùy ý xen kẽ với next vẫn đúng")

# Tính chất 3: lớp k vector tổng quát khớp oracle xoay vòng
for _ in range(300):
    k = random.randint(1, 5)
    vs = [[random.randint(0, 9) for _ in range(random.randint(0, 6))] for _ in range(k)]
    if not any(vs):
        continue
    for name in ("k_pointers",):
        assert drain(ZigzagIteratorK(*[list(v) for v in vs])) == oracle(vs)
    if k >= 2:
        assert drain(ZigzagQueue(*[list(v) for v in vs])) == oracle(vs)
        assert drain(ZigzagLazy(*[list(v) for v in vs])) == oracle(vs)
print("PASS tính chất: k vector (k = 1..5) khớp thứ tự xoay vòng")

# Phản ví dụ: zip cắt mất phần dư
print("zip làm mất phần dư:", [x for pair in zip([1, 2], [3, 4, 5, 6]) for x in pair], "| đúng:", oracle([[1, 2], [3, 4, 5, 6]]))

PASS tính chất: là hoán vị của hợp hai vector và đúng thứ tự xen kẽ


PASS tính chất: gọi hasNext tùy ý xen kẽ với next vẫn đúng


PASS tính chất: k vector (k = 1..5) khớp thứ tự xoay vòng
zip làm mất phần dư: [1, 3, 2, 4] | đúng: [1, 3, 2, 4, 5, 6]


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [22]:
random.seed(2026)
for trial in range(2000):
    v1 = [random.randint(-100, 100) for _ in range(random.randint(0, 15))]
    v2 = [random.randint(-100, 100) for _ in range(random.randint(0, 15))]
    if not v1 and not v2:
        continue
    expected = oracle([v1, v2])
    for name, cls in impls_two.items():
        r1, r2, r3 = run_two(cls, v1, v2)
        assert r1 == r2 == r3 == expected, (name, v1, v2)

for trial in range(40):                          # vector dài
    v1 = [random.randint(-10**9, 10**9) for _ in range(random.randint(0, 1000))]
    v2 = [random.randint(-10**9, 10**9) for _ in range(random.randint(1, 1000))]
    expected = oracle([v1, v2])
    for name, cls in impls_two.items():
        assert drain(cls(list(v1), list(v2))) == expected, (name, len(v1), len(v2))
print("PASS 2000 cặp vector ngẫu nhiên (3 kiểu gọi) và 40 cặp dài: mọi lớp khớp oracle độc lập")

PASS 2000 cặp vector ngẫu nhiên (3 kiểu gọi) và 40 cặp dài: mọi lớp khớp oracle độc lập


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Không bỏ qua vector đã hết (IndexError)

In [23]:
class SaiKhongBoQua:
    def __init__(self, v1, v2):
        self.vectors = [v1, v2]
        self.indexes = [0, 0]
        self.cur = 0

    def next(self):
        value = self.vectors[self.cur][self.indexes[self.cur]]      # sai: vector tới lượt có thể đã hết
        self.indexes[self.cur] += 1
        self.cur = 1 - self.cur
        return value

    def hasNext(self):
        return any(self.indexes[j] < len(self.vectors[j]) for j in range(2))


try:
    print(drain(SaiKhongBoQua([1, 2], [3, 4, 5, 6])))
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


**Vì sao sai:** khi `v1` hết nhưng `v2` còn, tới lượt `v1` code vẫn đọc `v1[indexes[0]]` ngoài biên. Phải **bỏ qua** vector đã hết (dời `cur`, hoặc chỉ giữ các vector còn phần tử trong hàng đợi).

### Lỗi 2: `hasNext` chỉ kiểm tra vector đang tới lượt

In [24]:
class SaiHasNextSom:
    def __init__(self, v1, v2):
        self.vectors = [v1, v2]
        self.indexes = [0, 0]
        self.cur = 0

    def next(self):
        j = self.cur if self.indexes[self.cur] < len(self.vectors[self.cur]) else 1 - self.cur
        value = self.vectors[j][self.indexes[j]]
        self.indexes[j] += 1
        self.cur = 1 - j
        return value

    def hasNext(self):
        return self.indexes[self.cur] < len(self.vectors[self.cur])        # sai: chỉ xét vector tới lượt


print(drain(SaiHasNextSom([1, 2], [3, 4, 5, 6])), "| đúng:", [1, 3, 2, 4, 5, 6])

[1, 3, 2, 4] | đúng: [1, 3, 2, 4, 5, 6]


**Vì sao sai:** khi vector tới lượt đã hết nhưng vector kia còn phần tử, `hasNext` trả `False` quá sớm. `hasNext` phải trả `True` nếu **bất kỳ** vector nào còn phần tử.

### Lỗi 3: Dùng `pop(0)` làm sửa đầu vào và chậm

In [25]:
class SaiPop0:
    def __init__(self, v1, v2):
        self.vectors = [v1, v2]
        self.cur = 0

    def hasNext(self):
        return any(self.vectors)

    def next(self):
        while not self.vectors[self.cur]:
            self.cur = 1 - self.cur
        value = self.vectors[self.cur].pop(0)                 # sai: sửa list gốc và pop(0) tốn O(n)
        self.cur = 1 - self.cur
        return value


a, b = [1, 2], [3, 4, 5, 6]
print(drain(SaiPop0(a, b)), "| đầu vào sau khi dùng:", a, b)

[1, 3, 2, 4, 5, 6] | đầu vào sau khi dùng: [] []


**Vì sao sai:** `pop(0)` **xóa** phần tử khỏi list gốc (người gọi mất dữ liệu) và mỗi lần tốn `O(độ dài)` vì dồn mọi phần tử. Hãy dùng **chỉ số** (hoặc `deque`) mà không sửa dữ liệu gốc.

### Lỗi 4: Dùng `zip` làm mất phần dư của vector dài hơn

In [26]:
class SaiZip:
    def __init__(self, v1, v2):
        self.flat = [x for pair in zip(v1, v2) for x in pair]             # sai: zip dừng ở vector ngắn nhất
        self.pos = 0

    def hasNext(self):
        return self.pos < len(self.flat)

    def next(self):
        value = self.flat[self.pos]
        self.pos += 1
        return value


print(drain(SaiZip([1, 2], [3, 4, 5, 6])), "| đúng:", [1, 3, 2, 4, 5, 6])

[1, 3, 2, 4] | đúng: [1, 3, 2, 4, 5, 6]


**Vì sao sai:** `zip` dừng khi dãy ngắn nhất hết nên `5, 6` (phần dư của `v2`) bị mất. Dùng `itertools.zip_longest` (có `fillvalue` rồi bỏ đi), hoặc quản lý bằng chỉ số/hàng đợi.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Bài thiết kế iterator: lưu **vị trí** (không sao chép dữ liệu); `hasNext` **không tiêu hao**; `next` an toàn bằng cách tự đảm bảo trạng thái.
- "Xen kẽ / quay vòng / round-robin nhiều nguồn": **hàng đợi các nguồn còn phần tử** (`popleft` rồi `append` nếu còn): `O(1)` mỗi thao tác, mở rộng tự nhiên cho `k` nguồn.
- Quay vòng chỉ số bằng `(cur + 1) % k`.
- Python lười: generator + **xem trước một phần tử** để cài `hasNext`.
- Đừng dùng `zip` khi độ dài khác nhau; dùng `zip_longest`.
- Test iterator: `while hasNext: next`, `hasNext` lặp lại, `next` không cần `hasNext`, đầu vào không bị sửa.

## 15. Biến thể của bài

1. **Cyclic iterator cho `k` vector** (follow up): xoay vòng.
2. **Iterator duyệt vector các vector** (Flatten 2D Vector, LeetCode 251).
3. **Iterator gộp nhiều danh sách đã sắp** (merge `k` list bằng heap).
4. **Iterator có thể `remove()` phần tử vừa trả.**
5. **Iterator hai chiều** (có `hasPrevious`/`previous`).

In [27]:
class Vector2D:
    """Biến thể 2 (LeetCode 251): trải phẳng danh sách các danh sách, lười, dùng hai chỉ số."""

    def __init__(self, vec):
        self.vec = vec
        self.i = self.j = 0

    def _advance(self):
        while self.i < len(self.vec) and self.j == len(self.vec[self.i]):
            self.i += 1
            self.j = 0

    def hasNext(self):
        self._advance()
        return self.i < len(self.vec)

    def next(self):
        self._advance()
        value = self.vec[self.i][self.j]
        self.j += 1
        return value


class SortedMergeIterator:
    """Biến thể 3: gộp k danh sách đã sắp bằng heap."""

    def __init__(self, *lists):
        import heapq
        self.heap = [(l[0], i, 0) for i, l in enumerate(lists) if l]
        heapq.heapify(self.heap)
        self.lists = lists
        self.heapq = heapq

    def hasNext(self):
        return bool(self.heap)

    def next(self):
        value, i, j = self.heapq.heappop(self.heap)
        if j + 1 < len(self.lists[i]):
            self.heapq.heappush(self.heap, (self.lists[i][j + 1], i, j + 1))
        return value


class BidirectionalIterator:
    """Biến thể 5."""

    def __init__(self, seq):
        self.seq, self.i = list(seq), 0

    def hasNext(self):
        return self.i < len(self.seq)

    def next(self):
        value = self.seq[self.i]
        self.i += 1
        return value

    def hasPrevious(self):
        return self.i > 0

    def previous(self):
        self.i -= 1
        return self.seq[self.i]


assert drain(Vector2D([[1, 2], [], [3], [], [4, 5, 6]])) == [1, 2, 3, 4, 5, 6]
assert drain(Vector2D([[], []])) == []
assert drain(SortedMergeIterator([1, 4, 7], [2, 5], [3, 6, 8])) == [1, 2, 3, 4, 5, 6, 7, 8]
bi = BidirectionalIterator([1, 2, 3])
assert [bi.next(), bi.next(), bi.previous(), bi.next(), bi.next()] == [1, 2, 2, 2, 3]
random.seed(15)
for _ in range(200):
    rows = [[random.randint(0, 9) for _ in range(random.randint(0, 5))] for _ in range(random.randint(0, 6))]
    assert drain(Vector2D(rows)) == [x for r in rows for x in r]
print("PASS biến thể: trải phẳng 2D, gộp danh sách đã sắp, iterator hai chiều")

PASS biến thể: trải phẳng 2D, gộp danh sách đã sắp, iterator hai chiều


## 16. So sánh các cách làm

| Cách | Ý tưởng | Khởi tạo | Mỗi thao tác | Space | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Trải phẳng sẵn | dựng `flat` rồi con trỏ | `O(n)` | `O(1)` | `O(n)` | đơn giản; oracle |
| 2. Hai con trỏ + biến lượt | `indexes`, `cur`, `hasNext` dời `cur` | `O(1)` | `O(k)` | `O(k)` | **cách nộp bài** |
| 3. Hàng đợi vector còn phần tử | `deque` `(vector, chỉ số)` | `O(k)` | `O(1)` | `O(k)` | **follow up** (`k` vector), tốt nhất |
| 4. `zip_longest` + xem trước | generator lười, bộ đệm 1 phần tử | `O(1)` | `O(1)` khấu hao | `O(k)` | phong cách Python |
| 5. `k` vector bằng con trỏ | tổng quát Cách 2 | `O(k)` | `O(k)` | `O(k)` | mở rộng bằng con trỏ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `drain(ZigzagIterator([1, 2], [3]))`, `drain(ZigzagIterator([], [5, 6]))`, `(2 + 1) % 3`, `(0 - 1) % 3`, và `list(zip([1, 2, 3], [4, 5]))`.

**Bài 2 (tính tay).** Với `v1 = [1, 2, 3]`, `v2 = [4]`: viết `cur`, `indexes` và giá trị trả về cho từng lần `next()`. Làm lại với `v1 = []`, `v2 = [7, 8]`.

**Bài 3 (điền TODO).** Hoàn thành `class ZigzagTodo` theo Cách 3 (hàng đợi).

In [28]:
class ZigzagTodo:
    def __init__(self, v1, v2):
        # TODO 1: self.queue = deque của các cặp (vector, 0) cho vector KHÔNG rỗng
        pass

    def next(self):
        # TODO 2: vector, i = popleft(); nếu i + 1 < len(vector) thì append((vector, i + 1))
        # TODO 3: trả vector[i]
        pass

    def hasNext(self):
        # TODO 4: trả True nếu hàng đợi không rỗng
        pass

# KHÔNG tạo đối tượng / gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `class CyclicIterator` cho `k` vector (follow up) bằng hàng đợi, nhận `*vectors`. Kiểm tra bằng ví dụ `[1,2,3]`, `[4,5,6,7]`, `[8,9]` cho `[1,4,8,2,5,9,3,6,7]`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `class PeekZigzag` thêm phương thức `peek()` trả phần tử kế mà không tiến lên (dựa trên Cách 3). (b) Viết `zigzag_weighted(v1, v2, a, b)` xen kẽ theo tỉ lệ: mỗi lượt lấy `a` phần tử từ `v1` rồi `b` phần tử từ `v2` (bỏ qua vector hết).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: xen kẽ theo cột, vector hết thì lấy tiếp vector kia; `%` quay vòng và cho kết quả không âm; `zip` dừng ở dãy ngắn. Bài 2: viết `indexes` và `cur` sau mỗi bước.
- **Mức 2 (vừa):** Bài 1: `[1, 3, 2]`, `[5, 6]`, `0`, `2`, `[(1, 4), (2, 5)]`. Bài 2: `[1,2,3]`/`[4]`: `1, 4, 2, 3` (sau khi `v2` hết chỉ còn `v1`); `[]`/`[7,8]`: `7, 8` (lượt đầu của `v1` rỗng nên bỏ qua). Bài 3: bốn TODO trong comment.
- **Mức 3 (sâu):** Bài 4: `deque((v, 0) for v in vectors if v)`; `next` lấy đầu, đọc, đưa lại nếu còn. Bài 5(a): giữ thêm biến `self.peeked`; hoặc nhìn `queue[0]` để lấy `vector[i]` mà không `popleft` (cách của Cách 3). Bài 5(b): vòng `for _ in range(a)` rồi `for _ in range(b)` với chỉ số.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert drain(ZigzagIterator([1, 2], [3])) == [1, 3, 2]
assert drain(ZigzagIterator([], [5, 6])) == [5, 6]
assert (2 + 1) % 3 == 0 and (0 - 1) % 3 == 2
assert list(zip([1, 2, 3], [4, 5])) == [(1, 4), (2, 5)]

# Bài 2
assert drain(ZigzagIterator([1, 2, 3], [4])) == [1, 4, 2, 3]
assert drain(ZigzagIterator([], [7, 8])) == [7, 8]

# Bài 3
class ZigzagTodoAns:
    def __init__(self, v1, v2):
        self.queue = deque((v, 0) for v in (v1, v2) if v)

    def next(self):
        vector, i = self.queue.popleft()
        if i + 1 < len(vector):
            self.queue.append((vector, i + 1))
        return vector[i]

    def hasNext(self):
        return bool(self.queue)

for vectors, expected in examples[:3]:
    assert drain(ZigzagTodoAns(*vectors)) == expected

# Bài 4
class CyclicIterator:
    def __init__(self, *vectors):
        self.queue = deque((v, 0) for v in vectors if v)

    def hasNext(self):
        return bool(self.queue)

    def next(self):
        v, i = self.queue.popleft()
        if i + 1 < len(v):
            self.queue.append((v, i + 1))
        return v[i]

assert drain(CyclicIterator([1, 2, 3], [4, 5, 6, 7], [8, 9])) == [1, 4, 8, 2, 5, 9, 3, 6, 7]

# Bài 5(a), 5(b)
class PeekZigzag(ZigzagQueue):
    def peek(self):
        vector, i = self.queue[0]
        return vector[i]

def zigzag_weighted(v1, v2, a, b):
    out, i, j = [], 0, 0
    while i < len(v1) or j < len(v2):
        for _ in range(a):
            if i < len(v1):
                out.append(v1[i])
                i += 1
        for _ in range(b):
            if j < len(v2):
                out.append(v2[j])
                j += 1
    return out

pz = PeekZigzag([1, 2], [3, 4])
assert pz.peek() == 1 and pz.next() == 1 and pz.peek() == 3 and pz.next() == 3 and pz.peek() == 2
assert zigzag_weighted([1, 2, 3, 4, 5], [10, 20], 2, 1) == [1, 2, 10, 3, 4, 20, 5]
assert zigzag_weighted([1, 2], [3, 4, 5, 6], 1, 1) == oracle([[1, 2], [3, 4, 5, 6]])
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (hai con trỏ + biến lượt; `next` gọi `hasNext` để an toàn)

```python
class ZigzagIterator:
    def __init__(self, v1: List[int], v2: List[int]):
        self.cur = 0
        self.size = 2
        self.indexes = [0] * self.size
        self.vectors = [v1, v2]

    def next(self) -> int:
        self.hasNext()
        vector = self.vectors[self.cur]
        index = self.indexes[self.cur]
        res = vector[index]
        self.indexes[self.cur] = index + 1
        self.cur = (self.cur + 1) % self.size
        return res

    def hasNext(self) -> bool:
        start = self.cur
        while self.indexes[self.cur] == len(self.vectors[self.cur]):
            self.cur = (self.cur + 1) % self.size
            if self.cur == start:
                return False
        return True
```

**Checklist**

- [ ] Lưu **chỉ số và lượt** (không sao chép dữ liệu, không sửa vector gốc, không `pop(0)`)?
- [ ] Bỏ qua vector đã hết (dời `cur` hoặc chỉ giữ vector còn phần tử trong hàng đợi)?
- [ ] `hasNext` trả `True` nếu **bất kỳ** vector còn phần tử và **không tiêu hao** phần tử?
- [ ] `next` an toàn kể cả khi vector đầu rỗng (gọi `hasNext` bên trong hoặc dùng hàng đợi)?
- [ ] Không dùng `zip` thay `zip_longest` khi độ dài khác nhau?
- [ ] Đã thử: `v1` rỗng, `v2` rỗng, hai vector cùng độ dài, `k = 3` (follow up)?
- [ ] Đã test nhiều kiểu gọi (`hasNext` lặp lại, `next` không cần `hasNext`) và đối chiếu oracle xoay vòng?
- [ ] Skeleton đúng `class ZigzagIterator` / `__init__(self, v1, v2)` / `next(self)` / `hasNext(self)`?